# Controlled-resolution re-run (v2) — reconciles paper Table 1b / Table 9 / Fig 7

**Goal.** Re-measure the controlled-resolution sweep on ImageNet with ONE run so that
Table 1b, Table 9, and Fig 7 share a single source. Uses the exact scoring of the
centerpiece AXIS3 harness (standardize each detector by `|(f - mu_clean)/sd_clean|`,
bicubic down/up-sample, FGSM eps=8/255), but with:
- `RES_N = 500` clean images (shuffled with SEED=42),
- all 7 detectors (HF-Energy, DCT-HFE, DFT-HFE, UPVR, O2, GaussianL1, PredL1),
- natural high-frequency energy measured on the same images.




In [ ]:
# ===================== [PREAMBLE] helpers (same conventions as centerpiece) =====================
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds='ImageNet'):
    mean=torch.tensor(IMGNET_MEAN).view(1,3,1,1).to(device); std=torch.tensor(IMGNET_STD).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds='ImageNet'):
    m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75):
    return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    patches=p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9)
    return patches.median(-1).values
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet',p)
    return out
print('[PREAMBLE] helpers ready; device=',device)


In [ ]:
# ===================== detectors (identical to centerpiece) =====================
def _gray255(b):
    w=torch.tensor([0.299,0.587,0.114],device=b.device).view(1,3,1,1)
    return (b*w).sum(1)

def feat_hfe(b):
    return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()
def feat_gl(b,bb,pp,glsig):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(b,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1)
        sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
    return (p0-p2).abs().sum(1).cpu().numpy()
def feat_upvr(b):
    a=b.round().clamp(0,255).to(torch.int32).cpu().numpy()
    N=a.shape[0]; out=np.zeros(N)
    for n in range(N):
        tot=a[n].size
        u=sum(len(np.unique(a[n,c])) for c in range(a.shape[1]))
        out[n]=u/tot
    return out
def feat_dct_hfe(b, B=8):
    from scipy.fft import dctn
    g=_gray255(b).cpu().numpy(); N,H,W=g.shape
    Hc,Wc=(H//B)*B,(W//B)*B; out=np.zeros(N)
    for n in range(N):
        img=g[n,:Hc,:Wc]; acc=0.0; cnt=0
        for i in range(0,Hc,B):
            for j in range(0,Wc,B):
                blk=dctn(img[i:i+B,j:j+B],norm='ortho')
                hf=blk.copy(); hf[:B//2,:B//2]=0.0
                acc+=float(np.sqrt(np.mean(hf**2))); cnt+=1
        out[n]=acc/max(cnt,1)
    return out
def feat_dft_hfe(b, ring_r=0.25):
    g=_gray255(b).cpu().numpy(); N,H,W=g.shape
    yy,xx=np.mgrid[0:H,0:W]; cy,cx=H/2.0,W/2.0
    r=np.sqrt(((yy-cy)/cy)**2+((xx-cx)/cx)**2); ring=(r>=ring_r).astype(np.float32)
    out=np.zeros(N)
    for n in range(N):
        sp=np.abs(np.fft.fftshift(np.fft.fft2(g[n])))
        out[n]=float((sp*ring).sum()/(ring.sum()+1e-8))
    return out
def feat_o2(b, bb, pp, n_noise=16, sigma255=32.0):
    with torch.no_grad():
        base=bb(pp(b)); y=base.argmax(1)
        acc=torch.zeros_like(base)
        for _ in range(n_noise):
            xn=(b+torch.randn_like(b)*sigma255).clamp(0,255); acc+=bb(pp(xn))
        noisy=acc/n_noise
        lo0=base-base.gather(1,y.view(-1,1)); lon=noisy-noisy.gather(1,y.view(-1,1))
        g=(lon-lo0).cpu().numpy()
    for n in range(g.shape[0]): g[n,int(y[n].item())]=-np.inf
    return g.max(1)
def fgsm(x, y, bb, pp, eps255=8.0):
    x=x.clone().to(device).requires_grad_(True)
    loss=F.cross_entropy(bb(pp(x)),y.to(device))
    g,=torch.autograd.grad(loss,x)
    return (x+eps255*g.sign()).clamp(0,255).detach()

def batched(fn_feat, X, bs=64):
    out=[]
    for i in range(0,len(X),bs): out.append(fn_feat(X[i:i+bs].to(device)))
    return np.concatenate(out)
def detector_features(X, bb, pp, glsig=1.0):
    return {
        'HF-Energy': batched(lambda b: feat_hfe(b), X),
        'DCT-HFE':   batched(lambda b: feat_dct_hfe(b), X),
        'DFT-HFE':   batched(lambda b: feat_dft_hfe(b), X),
        'UPVR':      batched(lambda b: feat_upvr(b), X),
        'O2':        batched(lambda b: feat_o2(b, bb, pp), X),
        'GaussianL1':batched(lambda b: feat_gl(b, bb, pp, glsig), X),
        'PredL1':    batched(lambda b: feat_predl1(b, bb, pp), X),
    }
print('[detectors] ready')


In [ ]:
# ===================== controlled-resolution sweep (v2) =====================
# CONFIG
RES_N     = 500                       # clean images (shuffled, SEED=42)
RES_RATIOS= [1.0, 1.4, 2.0, 3.5, 7.0] # native ... x7
GLSIG     = 1.0                       # GaussianL1 blur sigma for ImageNet (matches harness)

MX=find_mixed(); assert 'ImageNet' in MX, 'ImageNet mixed_dataset.pkl (eps8) required on the server'
bb=load_backbone('ImageNet'); pp=make_pp('ImageNet')

mixed=pickle.load(open(MX['ImageNet'],'rb'))
clean_all=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
rng=np.random.RandomState(SEED)
sel=rng.permutation(len(clean_all))[:RES_N]
Xc0=torch.cat([clean_all[i] for i in sel],0)
print('clean images used:', Xc0.shape[0])

with torch.no_grad():
    Y0=torch.cat([bb(pp(Xc0[i:i+64].to(device))).argmax(1).cpu() for i in range(0,len(Xc0),64)])

def nat_hf_energy(X, bs=64):
    # natural high-frequency energy of clean images: same |x - blur_{0.5}(x)| statistic, mean over set
    vals=[]
    for i in range(0,len(X),bs):
        b=X[i:i+bs].to(device); vals.append(feat_hfe(b))
    return float(np.mean(np.concatenate(vals)))

AXIS3={}; NATHF={}
for ratio in RES_RATIOS:
    small=max(32,int(round(224/ratio)))
    ds_=F.interpolate(Xc0,size=small,mode='bicubic',align_corners=False).clamp(0,255)
    us =F.interpolate(ds_,size=224,mode='bicubic',align_corners=False).clamp(0,255)
    # FGSM on the upsampled clean (target = its own predicted label), same as AXIS3
    adv=[]
    for i in range(0,len(us),64):
        adv.append(fgsm(us[i:i+64].to(device),Y0[i:i+64],bb,pp,eps255=8.0).cpu())
    Xa=torch.cat(adv,0)
    fc=detector_features(us, bb, pp, GLSIG); fa=detector_features(Xa, bb, pp, GLSIG)
    AXIS3[ratio]={}
    for name in fc:
        c=fc[name]; mu=c.mean(); sd=c.std()+1e-8
        an=lambda v: np.abs((v-mu)/sd)
        y=np.r_[np.zeros(len(c)),np.ones(len(fa[name]))]
        AXIS3[ratio][name]=round(float(roc_auc_score(y,np.r_[an(c),an(fa[name])])),4)
    NATHF[ratio]=round(nat_hf_energy(us),5)
    print(f'  ratio x{ratio}: HF-AUC={AXIS3[ratio]["HF-Energy"]:.4f}  natHF={NATHF[ratio]:.5f}')
print('[sweep] done')


In [ ]:
# ===================== save =====================
OUT='./controlled_resolution_v2'; os.makedirs(OUT,exist_ok=True)
payload={
  'config':{'RES_N':RES_N,'RES_RATIOS':RES_RATIOS,'GLSIG':GLSIG,'seed':SEED,
            'scoring':'abs((f-mu_clean)/sd_clean)','attack':'FGSM eps=8/255','interp':'bicubic'},
  'AXIS3':AXIS3,          # ratio -> {detector: AUROC}
  'natural_HF_energy':NATHF,
}
json.dump(payload, open(os.path.join(OUT,'controlled_resolution_v2.json'),'w'), indent=2)
print('saved ->', os.path.join(OUT,'controlled_resolution_v2.json'))

# quick preview tables (HF-Energy row + natural-HF, ordered x7..x1 like Table 9)
order=[7.0,3.5,2.0,1.4,1.0]
print('\nUpscale ratio :', '  '.join(f'x{r}' for r in order))
print('HF-Energy AUC :', '  '.join(f'{AXIS3[r]["HF-Energy"]:.3f}' for r in order))
print('Natural HF en.:', '  '.join(f'{NATHF[r]:.5f}' for r in order))
print('\nFull per-detector (native x1.0):')
for k,v in AXIS3[1.0].items(): print(f'   {k:11s} {v:.4f}')
